# Setup for SQL Queries

In [2]:
!pip install duckdb pandas faker numpy

Imports

In [3]:
import duckdb
import pandas as pd
import numpy as np
import random

from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

random.seed(42)
np.random.seed(42)

con = duckdb.connect()

Generate Users Table

In [4]:
N_USERS = 10000

countries = [
    "US",
    "UK",
    "Canada",
    "Germany",
    "France",
    "Australia"
]

channels = [
    "Organic",
    "Google Ads",
    "Facebook Ads",
    "Referral",
    "Email"
]

users = []

start_date = datetime(2024, 1, 1)

for user_id in range(1, N_USERS + 1):

    signup_date = start_date + timedelta(
        days=random.randint(0, 180)
    )

    users.append({
        "user_id": user_id,
        "signup_date": signup_date.date(),
        "country": random.choice(countries),
        "marketing_channel": random.choice(channels)
    })

users_df = pd.DataFrame(users)

users_df.head()

,user_id,signup_date,country,marketing_channel
0,1,2024-06-12,US,Organic
1,2,2024-03-11,UK,Google Ads
2,3,2024-02-05,Australia,Organic
3,4,2024-06-22,Australia,Email
4,5,2024-01-23,France,Referral


Generate Realistic Funnel Events

In [5]:
events = []
event_id = 1

for _, user in users_df.iterrows():

    user_id = user["user_id"]
    signup_date = pd.Timestamp(user["signup_date"])

    current_time = signup_date

    # signup event
    events.append({
        "event_id": event_id,
        "user_id": user_id,
        "event_time": current_time,
        "event_type": "signup"
    })
    event_id += 1

    n_sessions = random.randint(3, 20)

    for _ in range(n_sessions):

        current_time += timedelta(
            days=random.randint(0, 10),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )

        # view product
        events.append({
            "event_id": event_id,
            "user_id": user_id,
            "event_time": current_time,
            "event_type": "view_product"
        })
        event_id += 1

        if random.random() < 0.7:

            current_time += timedelta(minutes=random.randint(1, 60))

            events.append({
                "event_id": event_id,
                "user_id": user_id,
                "event_time": current_time,
                "event_type": "search"
            })
            event_id += 1

        if random.random() < 0.4:

            current_time += timedelta(minutes=random.randint(1, 60))

            events.append({
                "event_id": event_id,
                "user_id": user_id,
                "event_time": current_time,
                "event_type": "add_to_cart"
            })
            event_id += 1

        if random.random() < 0.2:

            current_time += timedelta(minutes=random.randint(1, 60))

            events.append({
                "event_id": event_id,
                "user_id": user_id,
                "event_time": current_time,
                "event_type": "purchase"
            })
            event_id += 1

events_df = pd.DataFrame(events)

print("Users:", len(users_df))
print("Events:", len(events_df))

Users: 10000
Events: 276538


Load into DuckDB

In [6]:
con.register("users_df", users_df)
con.register("events_df", events_df)

con.sql("""
CREATE OR REPLACE TABLE users AS
SELECT *
FROM users_df
""")

con.sql("""
CREATE OR REPLACE TABLE events AS
SELECT *
FROM events_df
""")

Inspect Data

In [7]:
con.sql("""
SELECT *
FROM users
LIMIT 5
""").df()

,user_id,signup_date,country,marketing_channel
0,1,2024-06-12,US,Organic
1,2,2024-03-11,UK,Google Ads
2,3,2024-02-05,Australia,Organic
3,4,2024-06-22,Australia,Email
4,5,2024-01-23,France,Referral


In [8]:
con.sql("""
SELECT *
FROM events
LIMIT 10
""").df()

,event_id,user_id,event_time,event_type
0,1,1,2024-06-12 00:00:00,signup
1,2,1,2024-06-20 13:58:00,view_product
2,3,1,2024-06-20 14:26:00,search
3,4,1,2024-06-20 15:16:00,add_to_cart
4,5,1,2024-06-26 07:11:00,view_product
5,6,1,2024-06-26 07:16:00,search
6,7,1,2024-07-02 13:18:00,view_product
7,8,1,2024-07-02 14:18:00,add_to_cart
8,9,1,2024-07-06 06:59:00,view_product
9,10,1,2024-07-06 07:11:00,search


# Query 1

Retention cohort: for each "signup week," what % of users were active in week 1, 2, 3?

### Explanation

This query calculates weekly user retention by cohort using a window function. First, users are grouped into cohorts based on their signup week using `DATE_TRUNC('week', signup_date)`. Next, for each user event, the query calculates `week_number`, which represents the number of weeks between the user's signup date and the event date. The data is then aggregated to count the number of distinct active users for each combination of `signup_week` and `week_number`.

To calculate retention, we need the size of each cohort, which is the number of users active in Week 0 (the signup week). Rather than creating a separate cohort-size table and joining it back to the aggregated results, the query uses a window function to retrieve the cohort size directly. The expression `CASE WHEN week_number = 0 THEN active_users END` returns the cohort size only for the Week 0 row and `NULL` for all other rows. The window function `MAX(...) OVER (PARTITION BY signup_week)` then evaluates this expression across all rows within the same signup cohort. Since the only non-null value in each partition is the Week 0 user count, `MAX()` returns the cohort size and propagates it to every row in that cohort.

Finally, retention is calculated as `100.0 * active_users / cohort_size`, producing the percentage of users from the original cohort who were active in each subsequent week. The `ROUND(..., 2)` function formats the result to two decimal places.


In [15]:
con.sql("""
WITH user_activity AS (

    SELECT
        u.user_id,
        -- rounds a date down to the beginning of a period
        DATE_TRUNC('week', u.signup_date) AS signup_week,
        -- Calculate Weeks Since Signup
        DATE_DIFF(
            'week',
            u.signup_date,
            CAST(e.event_time AS DATE)
        ) AS week_number

    FROM users u
    JOIN events e
        ON u.user_id = e.user_id

),

cohort_activity AS (

    SELECT
        signup_week,
        week_number,
        COUNT(DISTINCT user_id) AS active_users
    FROM user_activity
    GROUP BY signup_week, week_number

)

SELECT
    signup_week,
    week_number,
    active_users,
    ROUND(
        100.0 * active_users /
        MAX(
            CASE
                WHEN week_number = 0
                THEN active_users
            END
        ) OVER (PARTITION BY signup_week),
        2
    ) AS retention_pct
FROM cohort_activity
ORDER BY signup_week, week_number
""").df()

,signup_week,week_number,active_users,retention_pct
0,2024-01-01,0,384,100.00
1,2024-01-01,1,340,88.54
2,2024-01-01,2,331,86.20
3,2024-01-01,3,308,80.21
4,2024-01-01,4,285,74.22
...,...,...,...,...
532,2024-06-24,15,35,9.72
533,2024-06-24,16,23,6.39
534,2024-06-24,17,10,2.78
535,2024-06-24,18,3,0.83


# Query 2

Funnel: count users who completed step 1, then step 1→2, then 1→2→3.

### Explanation

This query performs a funnel analysis to measure how users progress through three stages of the product journey:

```text
view_product → search → add_to_cart
```

The goal is to count how many users reached each step of the funnel and to ensure that the actions occurred in the correct order.

The query begins by creating the `funnel_steps` Common Table Expression (CTE). For each user, it identifies the timestamp of their first `view_product`, `search`, and `add_to_cart` events. This is accomplished using conditional aggregation with `MIN(CASE WHEN ...)`. For example, the expression:

```sql
MIN(
    CASE
        WHEN event_type = 'view_product'
        THEN event_time
    END
)
```

returns the earliest product view event for each user. The same logic is applied to the search and add-to-cart events. After grouping by `user_id`, the CTE produces one row per user containing the first time they completed each funnel step.

The final query counts users who successfully completed each stage. The first count measures how many users viewed a product by checking whether `view_time` exists. The second count measures how many users searched after viewing a product using the condition `search_time > view_time`. The third count measures how many users added an item to their cart after searching using the condition `cart_time > search_time`.

The timestamp comparisons are important because they ensure that users completed the actions in the intended sequence. A user who searched before viewing a product, or added an item to the cart before searching, would not be counted in the later stages of the funnel even if they performed all three actions.



In [17]:
con.sql("""
WITH funnel_steps AS (

SELECT
    user_id,

    -- First product view
    MIN(
        CASE
            WHEN event_type = 'view_product'
            THEN event_time
        END
    ) AS view_time,

    -- First search
    MIN(
        CASE
            WHEN event_type = 'search'
            THEN event_time
        END
    ) AS search_time,

    -- First add to cart
    MIN(
        CASE
            WHEN event_type = 'add_to_cart'
            THEN event_time
        END
    ) AS cart_time

FROM events

GROUP BY user_id

)

SELECT

-- Step 1: Viewed a product
COUNT(
    CASE
        WHEN view_time IS NOT NULL
        THEN 1
    END
) AS step_1_view_product,

-- Step 2: Viewed a product and then searched
COUNT(
    CASE
        WHEN view_time IS NOT NULL
         AND search_time > view_time
        THEN 1
    END
) AS step_2_view_to_search,

-- Step 3: Viewed, searched, and then added to cart
COUNT(
    CASE
        WHEN view_time IS NOT NULL
         AND search_time > view_time
         AND cart_time > search_time
        THEN 1
    END
) AS step_3_view_to_search_to_cart

FROM funnel_steps;

""")


┌─────────────────────┬───────────────────────┬───────────────────────────────┐
│ step_1_view_product │ step_2_view_to_search │ step_3_view_to_search_to_cart │
│        int64        │         int64         │             int64             │
├─────────────────────┼───────────────────────┼───────────────────────────────┤
│               10000 │                  9981 │                          8200 │
└─────────────────────┴───────────────────────┴───────────────────────────────┘

# Query 3

"First event per user": using ROW_NUMBER to deduplicate.

### Explanation

This query identifies the first event recorded for each user using the `ROW_NUMBER()` window function. The problem can be viewed as a deduplication task because each user may have multiple events in the `events` table, but the goal is to return only one row per user.

The `ROW_NUMBER()` function assigns a unique rank to each event within a user. The clause:

```sql
PARTITION BY user_id
```

creates a separate partition for each user, meaning the ranking starts over for every user. The clause:

```sql
ORDER BY event_time
```

sorts the events chronologically within each user's partition.

The first event receives `rn = 1`, the second receives `rn = 2`, and so on.

The outer query then filters for:

```sql
WHERE rn = 1
```

which keeps only the earliest event for each user. The final result contains one row per user, including the timestamp and type of their first recorded activity.



In [19]:
con.sql("""

WITH ranked_events AS (


SELECT
    user_id,
    event_time,
    event_type,

    ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY event_time
    ) AS rn

FROM events


)

SELECT
user_id,
event_time AS first_event_time,
event_type AS first_event_type
FROM ranked_events
WHERE rn = 1
ORDER BY user_id;
""")


┌─────────┬─────────────────────┬──────────────────┐
│ user_id │  first_event_time   │ first_event_type │
│  int64  │    timestamp_ns     │     varchar      │
├─────────┼─────────────────────┼──────────────────┤
│       1 │ 2024-06-12 00:00:00 │ signup           │
│       2 │ 2024-03-11 00:00:00 │ signup           │
│       3 │ 2024-02-05 00:00:00 │ signup           │
│       4 │ 2024-06-22 00:00:00 │ signup           │
│       5 │ 2024-01-23 00:00:00 │ signup           │
│       6 │ 2024-01-09 00:00:00 │ signup           │
│       7 │ 2024-02-25 00:00:00 │ signup           │
│       8 │ 2024-06-03 00:00:00 │ signup           │
│       9 │ 2024-02-20 00:00:00 │ signup           │
│      10 │ 2024-04-17 00:00:00 │ signup           │
│       · │          ·          │   ·              │
│       · │          ·          │   ·              │
│       · │          ·          │   ·              │
│    9991 │ 2024-03-27 00:00:00 │ signup           │
│    9992 │ 2024-05-15 00:00:00 │ signup      

#  Query 4

Rolling 7-day average of a metric

### Explanation

This query calculates the daily number of purchases and then computes a 7-day rolling average using a window function.

The first step is the `daily_metrics` CTE, which aggregates purchase events by day:

```sql
SELECT
    CAST(event_time AS DATE) AS day,
    COUNT(*) AS purchases
```

The timestamp is converted to a date so that all purchases occurring on the same day are grouped together. The result is a table containing one row per day and the total number of purchases made on that day.

The rolling average is calculated using:

```sql
AVG(purchases) OVER (
    ORDER BY day
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
)
```

The `ORDER BY day` clause sorts the data chronologically. The window frame:

```sql
ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
```

tells SQL to include the current row plus the six previous rows, creating a 7-day window.

For example, on January 7th the average would be calculated using:

```text
Jan 1, Jan 2, Jan 3, Jan 4, Jan 5, Jan 6, Jan 7
```

On January 8th the window moves forward:

```text
Jan 2, Jan 3, Jan 4, Jan 5, Jan 6, Jan 7, Jan 8
```

This is why it is called a rolling or moving average: the calculation window continuously slides through the dataset as time progresses.

The `ROUND(..., 2)` function formats the result to two decimal places for readability.



In [21]:
## Rolling 7-Day Average of Daily Purchases

con.sql("""
WITH daily_metrics AS (

SELECT
    CAST(event_time AS DATE) AS day,
    COUNT(*) AS purchases

FROM events

WHERE event_type = 'purchase'

GROUP BY day

)

SELECT
day,
purchases,

ROUND(
    AVG(purchases) OVER (
        ORDER BY day
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ),
    2
) AS rolling_7_day_avg

FROM daily_metrics

ORDER BY day;
""")

┌────────────┬───────────┬───────────────────┐
│    day     │ purchases │ rolling_7_day_avg │
│    date    │   int64   │      double       │
├────────────┼───────────┼───────────────────┤
│ 2024-01-02 │         1 │               1.0 │
│ 2024-01-03 │         2 │               1.5 │
│ 2024-01-04 │         3 │               2.0 │
│ 2024-01-05 │         4 │               2.5 │
│ 2024-01-06 │         5 │               3.0 │
│ 2024-01-07 │        10 │              4.17 │
│ 2024-01-08 │        12 │              5.29 │
│ 2024-01-09 │        15 │              7.29 │
│ 2024-01-10 │        17 │              9.43 │
│ 2024-01-11 │        17 │             11.43 │
│     ·      │         · │                ·  │
│     ·      │         · │                ·  │
│     ·      │         · │                ·  │
│ 2024-10-19 │         6 │              2.57 │
│ 2024-10-20 │         3 │              2.71 │
│ 2024-10-22 │         2 │              2.71 │
│ 2024-10-25 │         2 │              2.86 │
│ 2024-10-27 

# Query 5

A query you deliberately analyze: describe in a comment why it would be slow on a 100M-row table and what you'd do about it.

### Explanation

This exercise focuses on query optimization rather than query correctness. The SQL statement returns all events that occurred during the year 2024, but it is intentionally written in a way that may perform poorly on very large datasets.

The filter condition:

```sql id="g0p4lc"
EXTRACT(YEAR FROM event_time) = 2024
```

requires the database to compute the year value for every row before it can evaluate the condition. On a small dataset this overhead is negligible, but on a table containing 100 million rows it can become expensive.

A common consequence is that the query planner chooses a sequential scan (full table scan), meaning every row in the table must be read and evaluated. Even if an index exists on the `event_time` column, wrapping the column inside a function can prevent the optimizer from using the index efficiently.

To diagnose the issue, the first step would be to inspect the execution plan using:

```sql id="84pc9y"
EXPLAIN
```

or

```sql id="fqln4j"
EXPLAIN ANALYZE
```

The execution plan reveals how the database intends to execute the query and whether expensive operations such as sequential scans, sorts, or joins are occurring.

A more efficient solution is to filter directly on the timestamp column using a date range:

```sql id="b2g3r5"
WHERE event_time >= '2024-01-01'
  AND event_time < '2025-01-01'
```

This approach is often referred to as a sargable predicate because it allows the database engine to leverage indexes and partition pruning more effectively. Instead of evaluating a function on every row, the database can directly locate the relevant range of values.

In a production environment, I would first examine the execution plan, verify whether an index exists on the filtering column, and determine whether the query is performing a full table scan. Depending on the results, I would consider query rewrites, indexing strategies, or partitioning to reduce the amount of data that must be scanned. The key point is that a query can be logically correct while still being inefficient at scale, and understanding execution plans is essential for diagnosing these performance issues.


In [24]:
## Query Performance Analysis

#This query returns all events that occurred in 2024.
con.sql("""
SELECT *
FROM events
WHERE EXTRACT(YEAR FROM event_time) = 2024;

-- A more efficient version would be:

SELECT *
FROM events
WHERE event_time >= '2024-01-01'
AND event_time < '2025-01-01';

""").df()

,event_id,user_id,event_time,event_type
0,1,1,2024-06-12 00:00:00,signup
1,2,1,2024-06-20 13:58:00,view_product
2,3,1,2024-06-20 14:26:00,search
3,4,1,2024-06-20 15:16:00,add_to_cart
4,5,1,2024-06-26 07:11:00,view_product
...,...,...,...,...
276533,276534,10000,2024-03-29 08:54:00,view_product
276534,276535,10000,2024-03-29 09:41:00,purchase
276535,276536,10000,2024-04-03 11:55:00,view_product
276536,276537,10000,2024-04-03 12:09:00,add_to_cart
